In [1]:
import pandas as pd

df = pd.read_csv("./bbc_news.csv")

df.head(3)

,title,pubDate,guid,link,description
0,Ukraine: Angry Zelensky vows to punish Russian...,"Mon, 07 Mar 2022 08:01:56 GMT",https://www.bbc.co.uk/news/world-europe-60638042,https://www.bbc.co.uk/news/world-europe-606380...,The Ukrainian president says the country will ...
1,War in Ukraine: Taking cover in a town under a...,"Sun, 06 Mar 2022 22:49:58 GMT",https://www.bbc.co.uk/news/world-europe-60641873,https://www.bbc.co.uk/news/world-europe-606418...,"Jeremy Bowen was on the frontline in Irpin, as..."
2,Ukraine war 'catastrophic for global food',"Mon, 07 Mar 2022 00:14:42 GMT",https://www.bbc.co.uk/news/business-60623941,https://www.bbc.co.uk/news/business-60623941?a...,One of the world's biggest fertiliser firms sa...


In [5]:
from spacy.cli import download

download("en_core_web_sm")

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [11]:
import spacy

nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

def preprocess(text):
    doc = nlp(text)
    return [token.lemma_.lower() for token in doc if not token.is_stop and not token.is_punct and token.is_alpha]

df["clean_desc"] = df.description.apply(preprocess)

df.head(3)


,title,pubDate,guid,link,description,clean_desc
0,Ukraine: Angry Zelensky vows to punish Russian...,"Mon, 07 Mar 2022 08:01:56 GMT",https://www.bbc.co.uk/news/world-europe-60638042,https://www.bbc.co.uk/news/world-europe-606380...,The Ukrainian president says the country will ...,"[ukrainian, president, say, country, forgive, ..."
1,War in Ukraine: Taking cover in a town under a...,"Sun, 06 Mar 2022 22:49:58 GMT",https://www.bbc.co.uk/news/world-europe-60641873,https://www.bbc.co.uk/news/world-europe-606418...,"Jeremy Bowen was on the frontline in Irpin, as...","[jeremy, bowen, frontline, irpin, resident, co..."
2,Ukraine war 'catastrophic for global food',"Mon, 07 Mar 2022 00:14:42 GMT",https://www.bbc.co.uk/news/business-60623941,https://www.bbc.co.uk/news/business-60623941?a...,One of the world's biggest fertiliser firms sa...,"[world, big, fertiliser, firm, say, conflict, ..."


In [17]:
import gensim
import gensim.corpora as corpora
from gensim.models.ldamodel import LdaModel

id2words = corpora.Dictionary(df.clean_desc)
corpus = [id2words.doc2bow(text) for text in df.clean_desc]

lda_model = LdaModel(corpus=corpus, id2word=id2words, num_topics=2, random_state=42)


print("\n🔹 LDA Topics:")
for idx, topic in lda_model.print_topics(-1):
    print(f"Topic {idx}: {topic}")


🔹 LDA Topics:
Topic 0: 0.010*"win" + 0.009*"england" + 0.009*"say" + 0.008*"world" + 0.007*"year" + 0.006*"manchester" + 0.006*"league" + 0.006*"final" + 0.006*"man" + 0.005*"city"
Topic 1: 0.017*"say" + 0.008*"people" + 0.007*"bbc" + 0.007*"uk" + 0.007*"year" + 0.005*"new" + 0.005*"government" + 0.005*"party" + 0.004*"police" + 0.004*"tell"


In [14]:
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()
tfidf = tfidf_vectorizer.fit_transform(df.description)

nmf_model = NMF(n_components=10, random_state=42)
nmf_topics = nmf_model.fit_transform(tfidf)


feature_names = tfidf_vectorizer.get_feature_names_out()

for idx, topic in enumerate(nmf_model.components_):
    top_words = [feature_names[i] for i in topic.argsort()[-5:]]
    print(f"Topic {idx}: {top_words}")

Topic 0: ['has', 'from', 'bbc', 'of', 'the']
Topic 1: ['days', 'seven', 'past', 'have', 'been']
Topic 2: ['world', 'their', 'cup', 'at', 'as']
Topic 3: ['get', 'from', 'up', 'the', 'to']
Topic 4: ['her', 'after', 'he', 'his', 'was']
Topic 5: ['say', 'they', 'people', 'are', 'and']
Topic 6: ['monday', 'friday', 'sunday', 'saturday', 'on']
Topic 7: ['were', 'london', 'two', 'the', 'in']
Topic 8: ['says', 'but', 'it', 'for', 'is']
Topic 9: ['election', 'the', 'says', 'be', 'will']


In [18]:
import pyLDAvis.gensim_models as gensimvis
import pyLDAvis

vis = gensimvis.prepare(lda_model, corpus, id2words)
pyLDAvis.save_html(vis, "lda.html")